# Calculate and Save Indices

In [1]:
### GENERAL SETUP
%matplotlib inline  
# this enables plotting within notebook

#import modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np   # basic math library  you will type np.$STUFF  e.g., np.cos(1)
import numpy.linalg as LA
from matplotlib.gridspec import GridSpec
import timeit
import cartopy.crs as ccrs
import datetime
import scipy.stats as stats # imports stats functions https://docs.scipy.org/doc/scipy/reference/stats.html
import cartopy.feature as cfeature
import cftime

## Functions

In [ ]:
def eof(ds_anom):
    # calculate the first and second EOFs/PCs for a given anomaly field using SVD
    ds_anom = ds_anom.where(np.isnan(ds_anom) == False, 0)
    ds_anom_std = ds_anom / ds_anom.std('time')
    ds_anom_std = ds_anom_std.where(np.isnan(ds_anom_std) == False, 0)
    time, lon, lat = ds_anom.time, ds_anom.lon, ds_anom.lat
    
    a,b,c = np.shape(ds_anom.values)  ## have axis sizes for later (a, b, c)
    Y_stand = ds_anom_std.values.reshape(a, b*c);
    Y = ds_anom.values.reshape(a, b*c);
    # print(a,b,c)
    # print(np.shape(Y_stand))
    # print(np.shape(Y))
    
    start_time = timeit.default_timer()
    u,s,v=LA.svd(Y_stand)  ## Barnes Chapter 3 Equation (65)
    elapsed = timeit.default_timer() - start_time
    # print('Time elapsed in LA SVD method: ',elapsed,' seconds')
    
    eof_num = 1
    e1_svd = (v[eof_num-1,:]).reshape(b,c)
    z1_svd = u[:,eof_num-1]*(s[eof_num-1]); z1_svd = (z1_svd-np.mean(z1_svd))/np.std(z1_svd)  
    pc1 = z1_svd
    eof1 = (1./np.size(Y,axis=0))*np.dot(np.transpose(z1_svd),Y)   ## Barnes Chapter 3 Equation (79)
    eof1_plot = eof1.reshape(b,c)  ### this is the reshaped eigenvector to plot
    eof1_plot = xr.DataArray(eof1_plot,dims=['lat','lon'],coords={'lat': lat,'lon': lon})
    pc1 = xr.DataArray(pc1,dims=['time'],coords={'time': time})
    
    eof_num = 2
    e1_svd = (v[eof_num-1,:]).reshape(b,c)
    z1_svd = u[:,eof_num-1]*(s[eof_num-1]); z1_svd = (z1_svd-np.mean(z1_svd))/np.std(z1_svd)  
    pc2 = z1_svd
    eof2 = (1./np.size(Y,axis=0))*np.dot(np.transpose(z1_svd),Y)   ## Barnes Chapter 3 Equation (79)
    eof2_plot = eof2.reshape(b,c)  ### this is the reshaped eigenvector to plot
    eof2_plot = xr.DataArray(eof2_plot,dims=['lat','lon'],coords={'lat': lat,'lon': lon})
    pc2 = xr.DataArray(pc2,dims=['time'],coords={'time': time})    

    eof_full = xr.merge([eof1_plot.to_dataset(name='EOF1'),eof2_plot.to_dataset(name='EOF2')])
    pc_full = xr.merge([pc1.to_dataset(name='PC1'),pc2.to_dataset(name='PC2')])

    return eof_full, pc_full

In [ ]:
def detrend(dat, dim, order):
    """ detrend dat along the axis dim with a polynomial of order n"""
    params = dat.polyfit(dim=dim, deg=order)
    fit = xr.polyval(dat[dim], params.polyfit_coefficients)
    dat = dat-fit
    return dat

## Load data

In [ ]:
# open the data!
ds = xr.open_dataset(' /glade/work/smogen/CVP_bifurcation/data/SSH.regrid.nc')['SSH']
ds['time'] = pd.date_range("1958-01", "2020-12", freq="MS")
# center on the Pacific
ds = ds.roll(lon=180,roll_coords=True)
ds['lon'] = np.arange(0.5,360.5,1)
ds = ds/100 # convert to meters for SSH

In [ ]:
# calculate the PDO/NPGO using PCA
ssh_fosi = ds
# remove seasonal climatology
ssh_fosi_anom = ssh_fosi.groupby('time.month') - ssh_fosi.groupby('time.month').mean()
ssh_fosi_anom = ssh_fosi_anom.sel(lat=slice(20,62),lon=slice(180,250))

ssh_fosi_anom_detr = detrend_linear(ssh_fosi_anom,'time',1)
## calculate EOFs
%time explained, eof_full, pc_full = eof(ssh_fosi_anom_detr)

strength_npgo = pc_full['PC2']
strength_pdo = pc_full['PC1']

In [ ]:
eof_full['EOF1'].to_netcdf('FOSI.PDO.EOF.nc')
eof_full['EOF2'].to_netcdf('FOSI.NPGO.EOF.nc')

strength_npgo.to_netcdf('FOSI.NPGO.nc')
strength_pdo.to_netcdf('FOSI.PDO.nc')

In [ ]:
# # PCA for SODA data

# # open the data!
# ds = xr.open_dataset('/glade/work/smogen/CVP_bifurcation/data/SODA3.15.2_NP_ssh_monmean_1980-2020.regrid.nc').ssh; 
# ds['time'] = pd.date_range('1980-01','2020-12',freq='MS')

# ssh_soda = ds
# # remove seasonal climatology
# ssh_soda_anom = ssh_soda.groupby('time.month') - ssh_soda.groupby('time.month').mean()
# ssh_soda_anom = ssh_soda_anom.sel(lat=slice(20,62),lon=slice(180,250))

# ssh_soda_anom_detr = detrend_linear(ssh_soda_anom,'time',1)
# ## calculate EOFs
# %time explained, eof_full, pc_full = eof(ssh_soda_anom_detr)

# strength_npgo2 = pc_full['PC2']
# strength_pdo2 = pc_full['PC1']

# strength_npgo2.to_netcdf('SODA.NPGO.nc')
# strength_pdo2.to_netcdf('SODA.PDO.nc')

In [ ]:
# CESM2 data

# CESM2-LE SSH
ssh = xr.open_dataset('/glade/derecho/scratch/smogen/CESM2/SSH.full.regrid.nc')['SSH']
ssh['time'] = pd.date_range('1950-01','2100-12',freq='MS')
ssh = ssh.roll(lon=180,roll_coords=True)
ssh['lon'] = np.arange(0.5,360.5,1)

# CESM2-LE UVEL
uvel_cesm2 = xr.open_dataset('/glade/derecho/scratch/smogen/CESM2/UVEL.full.regrid.nc')
uvel_cesm2['time'] = pd.date_range('1950-01','2100-12',freq='MS')
uvel_cesm2 = uvel_cesm2.roll(lon=180,roll_coords=True)
uvel_cesm2['lon'] = np.arange(0.5,360.5,1)

## Calculate Indices

In [ ]:
data = ssh_fosi # can change which dataset is used

In [ ]:
# load alternative datasets of SSH
cmems_reg = xr.open_dataset('cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1M-m_1775067968676.regrid.nc').sla
soda_reg = xr.open_dataset('SODA3.15.2_NP_ssh_monmean_1980-2020.regrid.nc').ssh; soda_reg['time'] = pd.date_range('1980-01','2020-12',freq='MS')
aviso = xr.open_dataset('zos_AVISO_L4_199210-201012.nc')['zos']; aviso['time'] = pd.date_range('1992-10','2010-12',freq='MS')

### Latitude of Core

In [ ]:
fosi_core = data.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon')

In [ ]:
fosi_core.to_dataset(name = 'core').to_netcdf('FOSI.Lat.Core.nc')

In [ ]:
cesm2_core = ssh.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon')
cesm2_core.to_dataset(name = 'core').to_netcdf('CESM2.Lat.Core.nc')

In [ ]:
soda_core = soda_reg.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon')
soda_core.to_dataset(name = 'core').to_netcdf('SODA.Lat.Core.nc')

In [ ]:
# Figure to compare across datasets
f, ax = plt.subplots(1,1,figsize=(10,4))
cmems_reg.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon').plot(color='purple',alpha=0.5,linewidth=.2)
soda_reg.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon').plot(color='red',alpha=0.5,linewidth=.2)
aviso.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon').plot(color='blue',alpha=0.5,linewidth=.2)

cmems_reg.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon').rolling(time=6,center=True).mean().plot(color='purple',label='CMEMS',alpha=1,linewidth=1.5)
soda_reg.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon').rolling(time=6,center=True).mean().plot(color='red',label='SODA',alpha=1,linewidth=1.5)
aviso.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat').mean('lon').rolling(time=6,center=True).mean().plot(color='blue',label='AVISO',alpha=1,linewidth=1.5)

fosi_core.plot(color='k',linewidth=0.1,alpha=0.5)
fosi_core.rolling(time=6,center=True).mean().plot(color='k',label='FOSI')

plt.ylabel('latitude')
plt.legend(fontsize=13)
plt.ylim(35,52)
plt.xlim('1980-01','2021-12')
plt.title('LoC comparison',fontsize=20)

f.savefig('SF1.compare.LOC.pdf')

### Latitude of Bifurcation

In [ ]:
## LATITUDE OF BIFURCATION - calculate and save
fosi_bifurcation = data.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat').mean('lon')

In [ ]:
fosi_bifurcation.to_dataset(name = 'bifurcation').to_netcdf('FOSI.218.228.bifurcation.nc')

In [ ]:
## LATITUDE OF BIFURCATION - calculate and save
cesm_bifurcation = ssh.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat').mean('lon')
cesm_bifurcation.to_dataset(name = 'bifurcation').to_netcdf('CESM2.218.228.bifurcation.nc')

In [ ]:
soda_core = soda_reg.sel(lat = slice(37,54),lon=slice(218,228).mean('lon')
soda_core.to_dataset(name = 'core').to_netcdf('SODA.Lat.Bifurcation.nc')

In [ ]:
# CESM2

In [ ]:
f, ax = plt.subplots(1,1,figsize=(10,4))
cmems_reg.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat').mean('lon').plot(color='purple',alpha=0.5,linewidth=.2)
soda_reg.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat').mean('lon').plot(color='red',alpha=0.5,linewidth=.2)
aviso.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat').mean('lon').plot(color='blue',alpha=0.5,linewidth=.2)

cmems_reg.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat').mean('lon').rolling(time=12,center=True).mean().plot(color='purple',label='CMEMS',alpha=1,linewidth=1.5)
soda_reg.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat').mean('lon').rolling(time=12,center=True).mean().plot(color='red',label='SODA',alpha=1,linewidth=1.5)
aviso.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat').mean('lon').rolling(time=12,center=True).mean().plot(color='blue',label='AVISO',alpha=1,linewidth=1.5)

fosi_bifurcation.sel(lon=slice(218,228)).mean('lon').plot(color='k',linewidth=0.1,alpha=0.5)
fosi_bifurcation.sel(lon=slice(218,228)).mean('lon').rolling(time=6,center=True).mean().plot(color='k',label='FOSI')

plt.ylabel('latitude')
plt.xlim('1980-01','2021-12')
# plt.title('Low Res. Latitude of Core: 180-200°E')
plt.ylabel('latitude')
plt.legend(fontsize=13)
plt.ylim(37,55)

plt.title('LoB comparison',fontsize=20)
f.savefig('SF2.compare.LOB.pdf',dpi = 500)

### sensitivity of Bifurcation

In [ ]:
import matplotlib.patches as mpatches
ds_def = ds.sel(time='2001-01').squeeze('time')
f, ax = plt.subplots(1,1, figsize=(14,5), subplot_kw=dict(projection=ccrs.PlateCarree(central_longitude=180)))

# im = ds_def.sel(lat = slice(20,62),lon = slice(178,245)).plot(cmap='RdBu_r',transform = ccrs.PlateCarree(),levels = np.arange(-0.5,0.51,0.01), extend = 'both', add_colorbar=False)
im = ds_def.differentiate('lat').sel(lat = slice(20,62),lon = slice(178,245)).plot(cmap='PRGn',transform = ccrs.PlateCarree(),levels = np.arange(-0.09,0.0905,0.0005), extend = 'both', add_colorbar=False)
# (ds_def.sel(lat = slice(35,54),lon=slice(180,200)).differentiate('lat').idxmin('lat')).plot(transform = ccrs.PlateCarree(),linewidth=1,color='k')
ds_def.sel(lat = slice(37,55),lon = slice(178,260)).differentiate('lat').idxmin('lat').sel(lon = slice(218,228)).plot(transform = ccrs.PlateCarree(),color='k',linewidth=2)


ax.add_feature(cfeature.LAND, color='k', zorder=3)
gl = ax.gridlines(crs=ccrs.PlateCarree(), linewidth=1, linestyle='--', color='black', alpha=0.3, draw_labels=True)
gl.top_labels = False
gl.right_labels = False

# ax.add_patch(mpatches.Rectangle(xy=[180, 35], width=20, height=19,
#                                     edgecolor='k',
#                                     facecolor='none',
#                                     transform=ccrs.PlateCarree(),linewidth=2)
#                  )

ax.add_patch(mpatches.Rectangle(xy=[218, 37], width=10, height=19,
                                    edgecolor='k',
                                    facecolor='none',
                                    transform=ccrs.PlateCarree(),linewidth=2)
                 )

ax.add_patch(mpatches.Rectangle(xy=[219, 37.1], width=2, height=18.8,
                                    edgecolor='grey',
                                    facecolor='none',
                                     transform=ccrs.PlateCarree(),linewidth=2)
                 )

ax.add_patch(mpatches.Rectangle(xy=[222, 37.1], width=2, height=18.8,
                                    edgecolor='grey',
                                    facecolor='none',
                                    transform=ccrs.PlateCarree(),linewidth=2)
                 )

ax.add_patch(mpatches.Rectangle(xy=[225, 37.1], width=2, height=18.8,
                                    edgecolor='grey',
                                    facecolor='none',
                                    transform=ccrs.PlateCarree(),linewidth=2)
                 )

f.subplots_adjust(right=1.1)
cbar_ax = f.add_axes([0.85, 0.12, 0.015, 0.750])
cbar = f.colorbar(im, cax=cbar_ax, ticks=[-0.08,0,0.08])
# cbar = f.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=10)

plt.title('LoB Sensitivity',fontsize=20)
f.savefig('SF2.LoB.sensitivity.map.pdf')

In [ ]:
fosi_bifurcation = data.sel(lat = slice(37,54),lon=slice(218,228)).differentiate('lat').idxmin('lat')

f, ax = plt.subplots(1,1,figsize=(12,7))

fosi_bifurcation.sel(lon=slice(218,228)).mean('lon').rolling(time=6,center=True).mean().plot(color='k',zorder=15,linewidth=2,label='132-142°W')

fosi_bifurcation.sel(lon=slice(219,221)).mean('lon').rolling(time=6,center=True).mean().plot(color='lightcoral',label='133-135°W')

fosi_bifurcation.sel(lon=slice(222,224)).mean('lon').rolling(time=6,center=True).mean().plot(color='red',label='136-138°W')

fosi_bifurcation.sel(lon=slice(226,228)).mean('lon').rolling(time=6,center=True).mean().plot(color='firebrick',label='139-141°W')

plt.legend(fontsize=15,loc='lower left', framealpha=1,facecolor='w')

plt.title('LoB Sensitivity',fontsize=20)
f.savefig('SF2.LoB.sensitivity.TS.pdf')

### Strength of Core

In [ ]:
data_uvel = uvel

In [ ]:
uvel = xr.open_dataset('/glade/work/smogen/CVP_bifurcation/data/UVEL.regrid.nc')['UVEL'].isel(z_t = 0)#.sel(z_t=slice(0,10000))
uvel = uvel.roll(lon=180,roll_coords=True)
uvel['lon'] = np.arange(0.5,360.5,1)
uvel['time'] = pd.date_range("1958-01", "2020-12", freq="MS")

# select region around the LoC
lat_min = np.round(grad_west,0) - 1
lat_max = np.round(grad_west,0) + 1

uvel_west = data_uvel.sel(lon = slice(180,200)).mean('lon')
strength = data_uvel.where(uvel_west.lat < lat_max).where(uvel_west.lat > lat_min).mean('lat')

In [ ]:
strength.to_dataset(name='strength').to_netcdf('FOSI.UVEL.strength.nc')

In [ ]:
# CESM2
# select region around the LoC
lat_min = np.round(cesm2_core,0) - 1
lat_max = np.round(cesm2_core,0) + 1

uvel_west = uvel.sel(lon = slice(180,200)).mean('lon')
strength_cesm2 = uvel_cesm2.where(uvel_west.lat < lat_max).where(uvel_west.lat > lat_min).mean('lat')

strength_cesm2.to_dataset(name='strength').to_netcdf('CESM2.UVEL.strength.nc')

In [ ]:
# Mer (from SODA 8/18/25)
mer = pd.read_csv('/glade/work/smogen/CVP_bifurcation/data/soda_npc_core_strength_indexes.csv')

year = mer['year']
month = mer['month']

df = pd.DataFrame({'year': year.values,
                   'month': month.values,
                   'day': (np.zeros_like(month.values) + 1)})

time = pd.to_datetime(df)

mer_lat = mer['core']
mer_speed = mer['speed']
mer_speeda = mer['speeda']
mer_ugeo = mer['ugeo']
mer_ugeoa = mer['ugeoa']

mer_index = xr.DataArray(mer_lat)
mer_index = mer_index.to_dataset()
mer_index = mer_index.assign(speed = mer_speed, speeda = mer_speeda, ugeo = mer_ugeo, ugeoa = mer_ugeoa)

mer_index = mer_index.rename({'dim_0':'time'})
mer_index['time'] = time.values

mer_index.to_netcdf('MER.Core.Str.nc')